In [1]:
PROJECT_ID = "qwiklabs-gcp-01-5fc8b912fc6a"
DATASET    = "fraud_detection"
LOCATION   = "US"
GCS_URI    = "gs://labs.roitraining.com/data-to-ai-workshop/weather_data.csv"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

In [2]:
raw_table = f"{PROJECT_ID}.{DATASET}.weather_data"

job_config = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV,
    skip_leading_rows=1,
    autodetect=True,
    write_disposition="WRITE_TRUNCATE",
)
client.load_table_from_uri(GCS_URI, raw_table, job_config=job_config).result()

print("Loaded rows:", client.get_table(raw_table).num_rows)

Loaded rows: 300


In [3]:
client.query(f"SELECT * FROM `{raw_table}` LIMIT 5").to_dataframe()


,date,city,state,temperature_f,wind_speed_mph,precipitation_in,barometric_pressure_inHg,humidity_percent,weather_condition
0,2025-02-21,Atlanta,GA,55.7,5.0,0.12,29.80,50.4,Cloudy
1,2025-02-26,Atlanta,GA,75.2,10.4,0.03,29.58,49.9,Cloudy
2,2025-03-01,Atlanta,GA,51.7,4.7,0.08,29.74,49.9,Cloudy
3,2025-03-05,Atlanta,GA,74.4,5.1,0.02,29.92,50.4,Cloudy
4,2025-03-10,Atlanta,GA,59.5,9.6,0.09,29.67,57.2,Cloudy


In [4]:
# make a connection that lets bigquery call gemini, then give it permission
!bq mk --connection --location=US --connection_type=CLOUD_RESOURCE gemini_conn 2>/dev/null

import json, subprocess
info = json.loads(subprocess.run(
    ["bq", "show", "--format=prettyjson", "--connection", f"{PROJECT_ID}.US.gemini_conn"],
    capture_output=True, text=True).stdout)
sa = info["cloudResource"]["serviceAccountId"]
print("service account:", sa)

# allow that service account to use vertex ai
!gcloud projects add-iam-policy-binding {PROJECT_ID} \
    --member="serviceAccount:{sa}" --role="roles/aiplatform.user" --condition=None

Connection 301800548063.us.gemini_conn successfully created
service account: bqcx-301800548063-srw0@gcp-sa-bigquery-condel.iam.gserviceaccount.com
Updated IAM policy for project [qwiklabs-gcp-01-5fc8b912fc6a].
bindings:
- members:
  - serviceAccount:service-301800548063@gcp-sa-vertex-nb.iam.gserviceaccount.com
  role: roles/aiplatform.colabServiceAgent
- members:
  - serviceAccount:service-301800548063@gcp-sa-aiplatform-vm.iam.gserviceaccount.com
  role: roles/aiplatform.notebookServiceAgent
- members:
  - serviceAccount:service-301800548063@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/aiplatform.serviceAgent
- members:
  - serviceAccount:bqcx-301800548063-srw0@gcp-sa-bigquery-condel.iam.gserviceaccount.com
  role: roles/aiplatform.user
- members:
  - serviceAccount:qwiklabs-gcp-01-5fc8b912fc6a@qwiklabs-gcp-01-5fc8b912fc6a.iam.gserviceaccount.com
  role: roles/bigquery.admin
- members:
  - serviceAccount:301800548063@cloudbuild.gserviceaccount.com
  role: roles/cloudbuild.bu

In [7]:
client.query(f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET}.gemini`
REMOTE WITH CONNECTION `{PROJECT_ID}.US.gemini_conn`
OPTIONS (endpoint = 'gemini-2.5-flash')
""").result()
print("gemini model ready")

gemini model ready


In [8]:
# ask gemini to write a public weather warning for each row, save to a new table
client.query(f"""
CREATE OR REPLACE TABLE `{PROJECT_ID}.{DATASET}.weather_reports` AS
SELECT * FROM ML.GENERATE_TEXT(
  MODEL `{PROJECT_ID}.{DATASET}.gemini`,
  (
    SELECT *,
      CONCAT('Write a short, friendly public weather warning based on this data: ',
             TO_JSON_STRING(t)) AS prompt
    FROM `{raw_table}` AS t
    LIMIT 20
  ),
  STRUCT(0.2 AS temperature, 1024 AS max_output_tokens, TRUE AS flatten_json_output)
)
""").result()
print("reports written")

reports written


In [9]:
client.query(f"""
SELECT prompt, ml_generate_text_llm_result
FROM `{PROJECT_ID}.{DATASET}.weather_reports`
LIMIT 5
""").to_dataframe()

,prompt,ml_generate_text_llm_result
0,"Write a short, friendly public weather warning...",Hey Atlanta! 👋\n\nGet ready for a **cloudy** d...
1,"Write a short, friendly public weather warning...",Hey Atlanta! 👋\n\nLooks like it's going to be ...
2,"Write a short, friendly public weather warning...",Hey Atlanta! 👋\n\nGet ready for a **cloudy** d...
3,"Write a short, friendly public weather warning...",Hey Atlanta! 👋\n\nJust a friendly heads-up for...
4,"Write a short, friendly public weather warning...",Hey Atlanta! ☁️\n\nGet ready for a **cloudy** ...
